# Logistic regression of `PW_0.5h`

Predict whether a failure occurs within the next 30 minutes from lagged sensor readings.

1. Read `PW_0.5h/train.csv` and `PW_0.5h/val.csv`.
2. Expand each series with `lag_window`. Each row keeps the last `READING_WINDOW` minutes of the sensors. `PW_0.5h` stays the label of the newest minute and is not used as a feature.
3. Hold out the last 20% of the lagged training rows as an eval set.
4. Tune logistic regression with `BayesSearchCV`. The search covers an L1 penalty, an L2 penalty, and an elastic-net penalty, along with `C`, `l1_ratio`, class weight, and the intercept. Each candidate is scored by expanding-window time-series cross-validation. Trials are ranked by validation F1.
5. Choose the F1 threshold on the eval holdout, then report F1 and AUC on `train.csv` and `val.csv`.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from skopt import BayesSearchCV
from skopt.space import Categorical, Real

from pipeline_functions import encode_oxygen_over_threshold, lag_window

DATA_DIR = Path("PW_0.5h")
TRAIN_PATH = DATA_DIR / "train.csv"
VAL_PATH = DATA_DIR / "val.csv"
TARGET = "PW_0.5h"

READING_WINDOW = 30
EVAL_FRACTION = 0.20
N_SPLITS = 5
N_TRIALS = 40
RANDOM_SEED = 42

# saga accepts L1, L2, and elastic net. The cap is a convergence budget, not a searched parameter.
MAX_ITER = 5000

## Load and lag

`READING_WINDOW` is 30 minutes, the same length as the failure horizon. `Oxygen over threshold` is mapped to 0 or 1 before lagging. Each file is then lagged on its own, in timestamp order. The oxygen lags stay numeric 0/1, which logistic regression uses directly.

In [2]:
def load_series(path):
    frame = pd.read_csv(path, parse_dates=["timestamp"])
    return frame.set_index("timestamp").sort_index()


def summarize(name, frame):
    labels = frame[TARGET].astype(int)
    print(
        f"{name}: {len(frame):,} rows, {frame.shape[1] - 1} features, "
        f"{int(labels.sum()):,} positives ({labels.mean():.2%}), "
        f"{frame.index.min()} to {frame.index.max()}"
    )


train_raw = encode_oxygen_over_threshold(load_series(TRAIN_PATH))
val_raw = encode_oxygen_over_threshold(load_series(VAL_PATH))
train = lag_window(train_raw, READING_WINDOW)
val = lag_window(val_raw, READING_WINDOW)

summarize("train.csv", train)
summarize("val.csv", val)
lag0_columns = [column for column in train.columns if column.endswith("_lag_0") or column == TARGET]
train.loc[:, lag0_columns].head(3)

train.csv: 40,325 rows, 120 features, 810 positives (2.01%), 2023-08-01 03:34:00 to 2023-08-29 03:38:00
val.csv: 34,315 rows, 120 features, 540 positives (1.57%), 2023-08-29 04:08:00 to 2023-09-22 00:02:00


,CMS air pressure_lag_0,Concentrazione ossigeno base_lag_0,Nitrogen pressure_lag_0,Oxygen over threshold_lag_0,PW_0.5h
timestamp,,,,,
2023-08-01 03:34:00,-0.578901,-0.986817,-1.394478,0.0,0
2023-08-01 03:35:00,-0.578901,-0.986817,-1.394478,0.0,0
2023-08-01 03:36:00,1.270593,-0.986817,-1.394478,0.0,0


## Eval holdout

The eval set is the last 20% of lagged `train.csv`. It is used to pick the F1 threshold. Time-series cross-validation uses only the earlier fit rows. `val.csv` stays out of fitting and tuning.

In [3]:
def features_and_labels(frame):
    labels = frame[TARGET].astype(int)
    features = frame.drop(columns=[TARGET])
    return features, labels


X_train, y_train = features_and_labels(train)
eval_start = int(len(train) * (1 - EVAL_FRACTION))
if eval_start <= 0 or eval_start >= len(train):
    raise ValueError("Training series is too short for the eval holdout")

X_fit, y_fit = X_train.iloc[:eval_start], y_train.iloc[:eval_start]
X_eval, y_eval = X_train.iloc[eval_start:], y_train.iloc[eval_start:]
feature_columns = list(X_fit.columns)

SCALE_POS_WEIGHT = float((y_fit == 0).sum() / (y_fit == 1).sum())
print(
    f"fit rows {len(X_fit):,} ({int(y_fit.sum())} positives), "
    f"eval rows {len(X_eval):,} ({int(y_eval.sum())} positives)"
)
print(f"eval starts at {X_eval.index[0]}")
print(f"scale_pos_weight {SCALE_POS_WEIGHT:.2f}")

fit rows 32,260 (600 positives), eval rows 8,065 (210 positives)
eval starts at 2023-08-23 13:14:00
scale_pos_weight 52.77


## Hyperparameter search

`BayesSearchCV` draws 40 settings and splits them across three penalties. `C` is the inverse penalty strength, so a smaller `C` is a stronger penalty. For elastic net, `l1_ratio` is the share of that penalty that is L1 and the rest is L2. `class_weight` is either uniform or `balanced`. `fit_intercept` is on or off.

A trial's score is the mean, over expanding-window `TimeSeriesSplit` folds, of the best F1 on that fold. The threshold for that F1 is chosen on the fold's validation rows. The eval holdout is not part of the search. The winning settings are refit on the fit rows.

In [ ]:
def best_f1(y_true, proba):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba)
    grid = np.linspace(0.01, 0.99, 99)
    predicted = proba[None, :] >= grid[:, None]
    positive = y_true == 1
    true_positive = (predicted & positive).sum(axis=1)
    false_positive = (predicted & ~positive).sum(axis=1)
    false_negative = (~predicted & positive).sum(axis=1)
    precision = np.divide(
        true_positive,
        true_positive + false_positive,
        out=np.zeros(len(grid)),
        where=(true_positive + false_positive) > 0,
    )
    recall = np.divide(
        true_positive,
        true_positive + false_negative,
        out=np.zeros(len(grid)),
        where=(true_positive + false_negative) > 0,
    )
    denom = precision + recall
    f1 = np.divide(2 * precision * recall, denom, out=np.zeros(len(grid)), where=denom > 0)
    return float(f1.max())


def validation_f1(estimator, features, labels):
    proba = estimator.predict_proba(features)[:, 1]
    return best_f1(labels, proba)


def penalty_space(penalty):
    space = {
        "penalty": Categorical([penalty]),
        "C": Real(1e-3, 1e3, prior="log-uniform"),
        "class_weight": Categorical([None, "balanced"]),
        "fit_intercept": Categorical([True, False]),
    }
    if penalty == "elasticnet":
        space["l1_ratio"] = Real(0.0, 1.0)
    return space


l1_trials = N_TRIALS // 3
l2_trials = N_TRIALS // 3
elastic_trials = N_TRIALS - l1_trials - l2_trials

print("Time-series folds:")
splitter = TimeSeriesSplit(n_splits=N_SPLITS)
for fold, (train_idx, valid_idx) in enumerate(splitter.split(X_fit), start=1):
    if y_fit.iloc[valid_idx].nunique() < 2:
        raise ValueError("A time-series fold has one validation class")
    print(
        f"  fold {fold}: train {len(train_idx):,} ({int(y_fit.iloc[train_idx].sum())} positives), "
        f"valid {len(valid_idx):,} ({int(y_fit.iloc[valid_idx].sum())} positives)"
    )
print(
    f"BayesSearchCV draws {l1_trials} L1, {l2_trials} L2, and {elastic_trials} elastic-net settings"
)

search = BayesSearchCV(
    LogisticRegression(
        solver="saga",
        max_iter=MAX_ITER,
        random_state=RANDOM_SEED,
    ),
    [
        (penalty_space("l1"), l1_trials),
        (penalty_space("l2"), l2_trials),
        (penalty_space("elasticnet"), elastic_trials),
    ],
    cv=splitter,
    scoring=validation_f1,
    n_jobs=-1,
    refit=True,
    random_state=RANDOM_SEED,
    verbose=1,
)
started = time.perf_counter()
search.fit(X_fit, y_fit)
final_model = search.best_estimator_
print(f"\nsearch finished in {time.perf_counter() - started:.0f}s")
print(f"best cv_f1={search.best_score_:.4f}")
print("selected parameters:")
print(dict(search.best_params_))

trial_results = (
    pd.DataFrame(
        [
            {**params, "cv_f1": score}
            for params, score in zip(
                search.cv_results_["params"],
                search.cv_results_["mean_test_score"],
            )
        ]
    )
    .sort_values("cv_f1", ascending=False)
    .reset_index(drop=True)
)
trial_results

Time-series folds:
  fold 1: train 5,380 (120 positives), valid 5,376 (60 positives)
  fold 2: train 10,756 (180 positives), valid 5,376 (120 positives)
  fold 3: train 16,132 (300 positives), valid 5,376 (60 positives)
  fold 4: train 21,508 (360 positives), valid 5,376 (120 positives)
  fold 5: train 26,884 (480 positives), valid 5,376 (120 positives)
BayesSearchCV draws 13 L1, 13 L2, and 14 elastic-net settings
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits


## Scores on `train.csv` and `val.csv`

The positive-class threshold is the value on a 0.01 grid that maximizes F1 on the eval holdout. AUC does not use that threshold. `train.csv` includes the fit rows, so its score is partly in-sample. The eval holdout chooses the F1 threshold. `val.csv` was not used to fit the model, choose hyperparameters, or choose the threshold.

In [ ]:
def predict_proba(features):
    return final_model.predict_proba(features.loc[:, feature_columns])[:, 1]


def f1_threshold(y_true, proba):
    grid = np.linspace(0.01, 0.99, 99)
    scores = [
        f1_score(y_true, proba >= threshold, zero_division=0)
        for threshold in grid
    ]
    best = int(np.argmax(scores))
    return float(grid[best]), float(scores[best])


def binary_scores(y_true, proba, threshold):
    y_true = np.asarray(y_true).astype(int)
    predicted = (proba >= threshold).astype(int)
    return {
        "rows": int(len(y_true)),
        "positives": int(y_true.sum()),
        "f1": float(f1_score(y_true, predicted, zero_division=0)),
        "auc": float(roc_auc_score(y_true, proba)),
    }


eval_proba = predict_proba(X_eval)
threshold, _ = f1_threshold(y_eval.to_numpy(), eval_proba)
print(f"F1 threshold from the eval holdout: {threshold:.2f}")

X_val, y_val = features_and_labels(val)
score_rows = [
    {"split": "train.csv", **binary_scores(y_train, predict_proba(X_train), threshold)},
    {"split": "val.csv", **binary_scores(y_val, predict_proba(X_val), threshold)},
    {"split": "train.csv fit rows", **binary_scores(y_fit, predict_proba(X_fit), threshold)},
    {"split": "train.csv eval holdout", **binary_scores(y_eval, eval_proba, threshold)},
]
score_table = pd.DataFrame(score_rows).set_index("split")
score_table

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import CalibrationDisplay
from sklearn.metrics import brier_score_loss

N_BINS = 10
STRATEGY = "uniform"


def reliability_table(y_true, proba, n_bins, strategy):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba, dtype=float)
    if strategy == "quantile":
        edges = np.percentile(proba, np.linspace(0, 100, n_bins + 1))
    else:
        edges = np.linspace(0.0, 1.0, n_bins + 1)
    # Same bin edges as sklearn.calibration.calibration_curve.
    bin_ids = np.searchsorted(edges[1:-1], proba)
    rows = []
    for bin_id in range(n_bins):
        in_bin = bin_ids == bin_id
        count = int(in_bin.sum())
        if count == 0:
            continue
        mean_predicted = float(proba[in_bin].mean())
        observed = float(y_true[in_bin].mean())
        rows.append(
            {
                "bin": f"{edges[bin_id]:.2f}–{edges[bin_id + 1]:.2f}",
                "rows": count,
                "positives": int(y_true[in_bin].sum()),
                "mean_predicted": mean_predicted,
                "observed_frequency": observed,
                "predicted_minus_observed": mean_predicted - observed,
            }
        )
    return pd.DataFrame(rows)


def failure_episodes(y_true):
    labels = pd.Series(np.asarray(y_true).astype(int))
    run_id = labels.ne(labels.shift(fill_value=0)).cumsum()
    runs = labels.groupby(run_id).agg(label="first", length="size")
    return runs.loc[runs["label"] == 1, "length"]


val_proba = predict_proba(X_val)
y_val_array = y_val.to_numpy()
bins = reliability_table(y_val_array, val_proba, N_BINS, STRATEGY)
episodes = failure_episodes(y_val_array)

display = CalibrationDisplay.from_predictions(
    y_val_array,
    val_proba,
    n_bins=N_BINS,
    strategy=STRATEGY,
    name="val.csv",
)
display.ax_.set_title("Reliability diagram on val.csv")
display.ax_.set_xlabel("Mean predicted probability")
display.ax_.set_ylabel("Observed failure frequency")
plt.show()

prevalence = float(np.mean(y_val_array))
mean_predicted = float(np.mean(val_proba))
ece = float(np.average(np.abs(bins["predicted_minus_observed"]), weights=bins["rows"]))
brier = float(brier_score_loss(y_val_array, val_proba))
brier_base = prevalence * (1.0 - prevalence)
overstated_share = float(bins.loc[bins["predicted_minus_observed"] > 0, "rows"].sum()) / len(y_val_array)
worst = bins.loc[bins["predicted_minus_observed"].abs().idxmax()]

print("On the diagonal, the mean predicted probability equals the observed failure frequency.")
print("Points below the diagonal give those minutes a higher chance of failure than they had.")
print(
    f"val.csv failure rate is {prevalence:.2%} "
    f"({int(y_val_array.sum())} positive minutes, {len(episodes)} episodes, "
    f"median episode length {episodes.median():.0f} minutes)."
)
print(
    f"Mean predicted probability is {mean_predicted:.2%}, "
    f"{mean_predicted / prevalence:.1f} times the failure rate."
)
if mean_predicted > prevalence:
    print("Across the whole validation series, the scores overstate how often failures occur.")
else:
    print("Across the whole validation series, the scores understate how often failures occur.")
print(
    f"{overstated_share:.1%} of minutes fall in bins below the diagonal."
)
print(
    f"Equal-width ECE is {ece:.3f}. "
    f"Brier score is {brier:.4f}. "
    f"Predicting the failure rate for every minute scores {brier_base:.4f}."
)
print(
    f"Largest gap is bin {worst['bin']}: "
    f"mean predicted {worst['mean_predicted']:.3f}, "
    f"observed frequency {worst['observed_frequency']:.3f} "
    f"({int(worst['positives'])} positives in {int(worst['rows']):,} minutes)."
)
if worst["predicted_minus_observed"] > 0:
    print(
        f"In that bin the score is {worst['predicted_minus_observed']:.3f} "
        f"above the failure rate, so the plotted point sits below the diagonal."
    )
else:
    print(
        f"In that bin the score is {-worst['predicted_minus_observed']:.3f} "
        f"below the failure rate, so the plotted point sits above the diagonal."
    )
print(
    f"The F1 threshold of {threshold:.2f} cuts these same scores. "
    f"A minute above that cutoff is not a minute with a {threshold:.0%} chance of failure."
)
print(
    f"The {len(y_val_array):,} validation minutes contain {len(episodes)} failure episodes, "
    f"so each bin frequency rests on far fewer events than its row count."
)
print()
print(
    bins.to_string(
        index=False,
        formatters={
            "mean_predicted": "{:.3f}".format,
            "observed_frequency": "{:.3f}".format,
            "predicted_minus_observed": "{:+.3f}".format,
        },
    )
)